## Import Libraries

In [35]:
import pandas as pd

## Load dataset

In [37]:
df1 = pd.read_csv("dataset_A.csv")
df2 = pd.read_csv("dataset_B.csv")

## Data Inspection

In [59]:
print(df1.info())
print(df2.info())



<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   company_id      100 non-null    int64 
 1   company_name    100 non-null    object
 2   parent_company  100 non-null    object
 3   auditor         100 non-null    object
dtypes: int64(1), object(3)
memory usage: 3.9+ KB
None
<class 'pandas.core.frame.DataFrame'>
Index: 110 entries, 0 to 114
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   company_id    110 non-null    int64 
 1   company_name  110 non-null    object
dtypes: int64(1), object(1)
memory usage: 2.6+ KB
None


In [60]:
print(df1.head())
print(df2.head())

   company_id  company_name parent_company   auditor
0        1001  Company_1001         Group2  Deloitte
1        1002  Company_1002         Group1        EY
2        1003  Company_1003         Group5       PwC
3        1004  Company_1004         Group4       PwC
4        1005  Company_1005         Group4        EY
   company_id  company_name
0        1001  Company_1001
1        1002  Company_1002
2        1003  Company_1003
3        1004  Company_1004
4        1005  Company_1005


## Data Cleaning

In [61]:
print(df1.isnull().sum())

company_id        0
company_name      0
parent_company    0
auditor           0
dtype: int64


In [62]:
df1['company_name'] = df1['company_name'].fillna('Unknown')
df1 = df1.dropna(subset = ['company_id'])

In [63]:
print(df2.isnull().sum())

company_id      0
company_name    0
dtype: int64


In [64]:
df2['company_name'] = df2['company_name'].fillna('Unknown')
df2 = df2.dropna(subset = ['company_id'])

In [65]:
df1 = df1.drop_duplicates()
df2 = df2.drop_duplicates()

In [72]:
# Convert to lowercase
df1['company_name'] = df1['company_name'].str.lower()
df2['company_name'] = df2['company_name'].str.lower()

# Remove extra spaces
df1['company_name'] = df1['company_name'].str.strip()
df2['company_name'] = df2['company_name'].str.strip()

In [73]:
df1['company_name'] = df1['company_name'].str.replace("limited", "ltd")
df2['company_name'] = df2['company_name'].str.replace("limited", "ltd")

## Reconciliation (Compring Datasets)

## Merge Both DataSets

In [74]:
merged = pd.merge(df1,df2, on = 'company_id', how = 'outer', indicator = True)

## Find Mismatches

In [75]:
mismatch = merged[merged['_merge'] != 'both']

print(mismatch)

     company_id company_name_x parent_company auditor company_name_y  \
100        1101            NaN            NaN     NaN     newco_1101   
101        1102            NaN            NaN     NaN     newco_1102   
102        1103            NaN            NaN     NaN     newco_1103   
103        1104            NaN            NaN     NaN     newco_1104   
104        1105            NaN            NaN     NaN     newco_1105   
105        1106            NaN            NaN     NaN     newco_1106   
106        1107            NaN            NaN     NaN     newco_1107   
107        1108            NaN            NaN     NaN     newco_1108   
108        1109            NaN            NaN     NaN     newco_1109   
109        1110            NaN            NaN     NaN     newco_1110   

         _merge  
100  right_only  
101  right_only  
102  right_only  
103  right_only  
104  right_only  
105  right_only  
106  right_only  
107  right_only  
108  right_only  
109  right_only  


In [76]:
## Identifing Inconsistencies
name_mismatch = merged[
    (merged['company_name_x'] != merged['company_name_y']) &
    (merged['_merge'] == 'both')
]

print(name_mismatch)

    company_id company_name_x parent_company   auditor    company_name_y  \
9         1010   company_1010        Group10      KPMG  company_1010 ltd   
17        1018   company_1018        Group10       PwC  company_1018 ltd   
23        1024   company_1024         Group4      KPMG  company_1024 ltd   
24        1025   company_1025         Group8        EY  company_1025 ltd   
35        1036   company_1036         Group2        EY  company_1036 ltd   
54        1055   company_1055         Group4  Deloitte  company_1055 ltd   
57        1058   company_1058         Group4  Deloitte  company_1058 ltd   
59        1060   company_1060         Group2        EY  company_1060 ltd   
68        1069   company_1069         Group6        EY  company_1069 ltd   

   _merge  
9    both  
17   both  
23   both  
24   both  
35   both  
54   both  
57   both  
59   both  
68   both  


## Simulate Secondary Validation

In [77]:
reference = pd.read_csv("reference_data.csv")

validated = pd.merge(df1, reference, on='company_id', how='left')

# Find records not validated
not_valid = validated[validated['reference_flag'].isnull()]

print(not_valid)

    company_id  company_name parent_company   auditor reference_flag
17        1018  company_1018        Group10       PwC            NaN
22        1023  company_1023         Group7        EY            NaN
23        1024  company_1024         Group4      KPMG            NaN
28        1029  company_1029         Group3      KPMG            NaN
29        1030  company_1030         Group7        EY            NaN
31        1032       unknown         Group5      KPMG            NaN
34        1035  company_1035         Group6  Deloitte            NaN
42        1043  company_1043         Group5        EY            NaN
44        1045  company_1045         Group8  Deloitte            NaN
50        1051  company_1051         Group5      KPMG            NaN
53        1054  company_1054        Group10       PwC            NaN
57        1058  company_1058         Group4  Deloitte            NaN
60        1061  company_1061         Group4       PwC            NaN
65        1066  company_1066      

## Risk Detection Logic

In [78]:
risk = df1.groupby(['parent_company', 'auditor']).size().reset_index(name='count')

risk_cases = risk[risk['count'] > 1]

print(risk_cases)

   parent_company   auditor  count
0          Group1  Deloitte      3
1          Group1        EY      3
2          Group1       PwC      3
4         Group10      KPMG      4
5         Group10       PwC      3
6          Group2  Deloitte      6
7          Group2        EY      5
8          Group2      KPMG      2
10         Group3  Deloitte      2
12         Group3      KPMG      5
13         Group4  Deloitte      5
14         Group4        EY      3
15         Group4      KPMG      5
16         Group4       PwC      3
17         Group5  Deloitte      3
19         Group5      KPMG      4
20         Group5       PwC      3
21         Group6  Deloitte      5
23         Group6      KPMG      2
24         Group6       PwC      3
25         Group7        EY      3
26         Group7      KPMG      3
27         Group7       PwC      3
30         Group8      KPMG      2
31         Group8       PwC      2
34         Group9      KPMG      3
35         Group9       PwC      3


## Create Report

In [79]:
with pd.ExcelWriter("compliance_report.xlsx") as writer:
    mismatch.to_excel(writer, sheet_name="Mismatches", index=False)
    name_mismatch.to_excel(writer, sheet_name="Name Issues", index=False)
    not_valid.to_excel(writer, sheet_name="Validation Issues", index=False)
    risk_cases.to_excel(writer, sheet_name="Risk Cases", index=False)